In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import itertools
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import pandas as pd
from keras.utils.data_utils import get_file

In [ ]:
# Downloading training and test sets to local drive
try:
    training_set_path = get_file('KDDTrain%2B.csv', origin='https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.csv')
except:
    print('Error downloading')
    raise


try:
    test_set_path = get_file('KDDTest%2B.csv', origin='https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.csv')
except:
    print('Error downloading')
    raise
training_df = pd.read_csv(training_set_path, header=None)
testing_df = pd.read_csv(test_set_path, header=None)




In [ ]:
training_df.head()

In [ ]:
testing_df.head()

In [ ]:
columns = [
    'duration',
    'protocol_type',
    'service',
    'flag',
    'src_bytes',
    'dst_bytes',
    'land',
    'wrong_fragment',
    'urgent',
    'hot',
    'num_failed_logins',
    'logged_in',
    'num_compromised',
    'root_shell',
    'su_attempted',
    'num_root',
    'num_file_creations',
    'num_shells',
    'num_access_files',
    'num_outbound_cmds',
    'is_host_login',
    'is_guest_login',
    'count',
    'srv_count',
    'serror_rate',
    'srv_serror_rate',
    'rerror_rate',
    'srv_rerror_rate',
    'same_srv_rate',
    'diff_srv_rate',
    'srv_diff_host_rate',
    'dst_host_count',
    'dst_host_srv_count',
    'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate',
    'dst_host_srv_serror_rate',
    'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate',
    'outcome',
    'difficulty'
]
training_df.columns = columns
testing_df.columns = columns

In [ ]:
print("Training set has {} rows.".format(len(training_df)))
print("Testing set has {} rows.".format(len(testing_df)))

In [ ]:
training_outcomes=training_df["outcome"].unique()
testing_outcomes=testing_df["outcome"].unique()
print("The training set has {} possible outcomes \n".format(len(training_outcomes)) )
print(", ".join(training_outcomes)+".")
print("\nThe testing set has {} possible outcomes \n".format(len(testing_outcomes)))
print(", ".join(testing_outcomes)+".")

In [ ]:
# A list ot attack names that belong to each general attack type
dos_attacks=["snmpgetattack","back","land","neptune","smurf","teardrop","pod","apache2","udpstorm","processtable","mailbomb"]
r2l_attacks=["snmpguess","worm","httptunnel","named","xlock","xsnoop","sendmail","ftp_write","guess_passwd","imap","multihop","phf","spy","warezclient","warezmaster"]
u2r_attacks=["sqlattack","buffer_overflow","loadmodule","perl","rootkit","xterm","ps"]
probe_attacks=["ipsweep","nmap","portsweep","satan","saint","mscan"]

# Our new labels
classes=["Normal","Dos","R2L","U2R","Probe"]

#Helper function to label samples to 5 classes
def label_attack (row):
    if row["outcome"] in dos_attacks:
        return classes[1]
    if row["outcome"] in r2l_attacks:
        return classes[2]
    if row["outcome"] in u2r_attacks:
        return classes[3]
    if row["outcome"] in probe_attacks:
        return classes[4]
    return classes[0]


#We combine the datasets temporarily to do the labeling
test_samples_length = len(testing_df)
df=pd.concat([training_df,testing_df])
df["Class"]=df.apply(label_attack,axis=1)


# The old outcome field is dropped since it was replaced with the Class field, the difficulty field will be dropped as well.
df=df.drop("outcome",axis=1)
df=df.drop("difficulty",axis=1)

# we again split the data into training and test sets.
type_testing_df = testing_df
training_df= df.iloc[:-test_samples_length, :]
testing_df= df.iloc[-test_samples_length:,:]

In [ ]:
training_outcomes=training_df["Class"].unique()
testing_outcomes=testing_df["Class"].unique()
print("The training set has {} possible outcomes \n".format(len(training_outcomes)) )
print(", ".join(training_outcomes)+".")
print("\nThe testing set has {} possible outcomes \n".format(len(testing_outcomes)))
print(", ".join(testing_outcomes)+".")

In [ ]:
# Helper function for scaling continous values
def minmax_scale_values(training_df,testing_df, col_name):
    scaler = MinMaxScaler()
    scaler = scaler.fit(training_df[col_name].values.reshape(-1, 1))
    train_values_standardized = scaler.transform(training_df[col_name].values.reshape(-1, 1))
    training_df[col_name] = train_values_standardized
    test_values_standardized = scaler.transform(testing_df[col_name].values.reshape(-1, 1))
    testing_df[col_name] = test_values_standardized


#Helper function for one hot encoding
def encode_text(training_df,testing_df, name):
    training_set_dummies = pd.get_dummies(training_df[name])
    testing_set_dummies = pd.get_dummies(testing_df[name])
    for x in training_set_dummies.columns:
        dummy_name = "{}_{}".format(name, x)
        training_df[dummy_name] = training_set_dummies[x]
        if x in testing_set_dummies.columns :
            testing_df[dummy_name]=testing_set_dummies[x]
        else :
            testing_df[dummy_name]=np.zeros(len(testing_df))
    training_df.drop(name, axis=1, inplace=True)
    testing_df.drop(name, axis=1, inplace=True)


sympolic_columns=["protocol_type","service","flag"]
label_column="Class"
for column in df.columns :
    if column in sympolic_columns:
        encode_text(training_df,testing_df,column)
    elif not column == label_column:
        minmax_scale_values(training_df,testing_df, column)

In [ ]:
training_df.head(5)

In [ ]:
training_df.columns

In [ ]:
# Assuming 'data' is the DataFrame
column_names = training_df.columns.tolist()

# Display all column names
for name in column_names:
    print(name)
np.save('/content/drive/MyDrive/Colab Notebooks/xNIDS/Data/kdd_after_features.npy', column_names)

In [ ]:
x,y=training_df,training_df.pop("Class").values
x=x.values
x_test,y_test=testing_df,testing_df.pop("Class").values
x_test=x_test.values
y0=np.ones(len(y),np.int8)
y0[np.where(y==classes[0])]=0
y0_test=np.ones(len(y_test),np.int8)
y0_test[np.where(y_test==classes[0])]=0
input_shape = x.shape[1]

In [ ]:
x_test.shape[0]

In [ ]:
x_test.shape[-1]

In [ ]:

from tensorflow import keras
from keras.layers import LSTM, Input, Dense, Dropout
from keras.models import Model, Sequential

# Reshape the training and test data
x_train = np.reshape(x, (x.shape[0], 1, x.shape[-1]))
x_test = np.reshape(x_test, (x_test.shape[0], 1, x_test.shape[-1]))

# LSTM requirements
lst = Sequential()

# Input layer and LSTM layer with 50 neurons
lst.add(LSTM(50, batch_input_shape=(1, 1, x.shape[-1]), stateful=True, return_sequences=True))
lst.add(Dropout(0.2))  # Dropout layer with 20% dropout rate

# Additional LSTM layer with 50 neurons
lst.add(LSTM(10))
lst.add(Dropout(0.2))  # Dropout layer with 20% dropout rate
# Output layer with sigmoid activation
lst.add(Dense(1, activation='sigmoid'))

lst.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
lst.summary()

# Training the model with stateful LSTM
for epoch in range(2):
    history = lst.fit(x_train, y0, epochs=1, batch_size=1, validation_split=0.2, shuffle=False)
    lst.reset_states()

test_results = lst.evaluate(x_test, y0_test, batch_size=1, verbose=1)
print(f'Test results - Loss: {test_results[0]} - Accuracy: {test_results[1] * 100}%')

# Save the model
model_path = "/content/drive/MyDrive/Colab Notebooks/xNIDS/Models/lstm_history_model.h5"
lst.save(model_path)
print("Model saved.")

# Load the model
new_model = keras.models.load_model(model_path)
new_model.summary()
print("Model loaded.")

import matplotlib.pyplot as plt

# Plot accuracy vs epoch of train and test dataset
#plt.plot(history.history['accuracy'])
#plt.plot(history.history['val_accuracy'])
#plt.title("Plot of accuracy vs epoch for train and test dataset")
#plt.ylabel('accuracy')
#plt.xlabel('epoch')
#plt.legend(['train', 'test'], loc='best')
#plt.show()


In [ ]:
# Prepare data for explanations.
# Make predictions
predicted_probabilities = lst.predict(x_test,batch_size=1)
predicted_labels = (predicted_probabilities >= 0.5).astype(int)

rnn_false_positives = []  # Store indices of false positives
rnn_false_negatives = []  # Store indices of false negatives
for i in range(len(predicted_labels)):
    if predicted_labels[i][0] != y0_test[i]:
        if predicted_labels[i][0] == 1:  # False positive
            rnn_false_positives.append(i)
        else:
           rnn_false_negatives.append(i)

In [ ]:
rnn_false_positives

In [ ]:
#rnn_false_negatives

In [ ]:
kdd_selected_fp_rows = type_testing_df.loc[19000:19114]
kdd_selected_fp_rows_122 = pd.DataFrame(x_test[19000:19115].reshape(115,122))
kdd_selected_fp_rows.to_csv('/content/drive/MyDrive/Colab Notebooks/xNIDS/Data/kdd_history_selected_fp_rows.csv', index=True)
kdd_selected_fp_rows_122.to_csv('/content/drive/MyDrive/Colab Notebooks/xNIDS/Data/kdd_history_selected_fp_rows_122.csv', index=True)
# 19114

In [ ]:
kdd_selected_fp_rows

In [ ]:
predicted_probabilities[19114]

In [ ]:
idx = 19114

In [ ]:
y0_test[idx]

In [ ]:
lst.predict(x_test[idx].reshape(1,1,-1))

In [ ]:
lst.predict(x_test[idx-1:idx+1],batch_size=1)

In [ ]:
lst.predict(x_test[idx-3:idx+1],batch_size=1)

In [ ]:
lst.predict(x_test[idx-7:idx+1],batch_size=1)

In [ ]:
lst.predict(x_test[idx-15:idx+1],batch_size=1)

In [ ]:
#kdd_selected_fp_rows